# Pengolahan Data Budget untuk Supabase

Notebook ini dipakai untuk membaca data, memeriksa data budget, lalu menyiapkan CSV untuk database. Jalankan dari atas ke bawah.

**Folder input:** `data/`  
**Hasil budget valid:** `data/ready/`  
**Hasil CSV tabel:** `data/supabase_import/`

## STEP 01 — Load Data

**Tujuan:** Baca data budget dan tiga data master dari folder `data`. Pastikan semua file berhasil dibuka.

In [1]:
# step 01 - load data
# untuk membaca 4 sumber data
import pandas as pd
from pathlib import Path

# 1. lokasi folder data
DATA_DIR = Path("data")

# 2. Load data budget
df_budget = pd.read_excel(
    DATA_DIR / "clean-budget.xlsx",
    sheet_name="budget_detail",
    dtype={
        "source_entity": "string",
        "biro_code": "string",
        "funds_center_code": "string",
        "commitment_item_code": "string"
    }
)

# 3. Master CI
df_master_ci = pd.read_excel(
    DATA_DIR / "data-master.xlsx",
    sheet_name="Master CI",
    dtype="string"
)

# 4. Master FC
df_master_fc = pd.read_excel(
    DATA_DIR / "data-master.xlsx",
    sheet_name="Master FC",
    dtype="string"
)

# 5. Master Fund hasil cleaning
df_master_fund = pd.read_csv(
    DATA_DIR / "master-fund clean.csv",
    dtype="string"
)

# 6. status pembacaan
datasets = {
    "Budget Detail": df_budget,
    "Master CI": df_master_ci,
    "Master FC": df_master_fc,
    "Master Fund": df_master_fund
}

print("=== STEP 01: LOAD DATA ===\n")

for name, df in datasets.items():
    print(f"{name}")
    print(f"  Total baris : {len(df):,}")
    print(f"  Total kolom : {len(df.columns)}")
    print("-" * 40)

print("\nSemua data berhasil dimuat.")

=== STEP 01: LOAD DATA ===

Budget Detail
  Total baris : 7,504
  Total kolom : 17
----------------------------------------
Master CI
  Total baris : 1,499
  Total kolom : 3
----------------------------------------
Master FC
  Total baris : 1,776
  Total kolom : 3
----------------------------------------
Master Fund
  Total baris : 334
  Total kolom : 3
----------------------------------------

Semua data berhasil dimuat.


## STEP 02 — Cek Struktur Data

**Tujuan:** Lihat nama kolom, tipe data, jumlah data kosong, dan beberapa contoh baris.

In [2]:
# step 02 - structural profiling
# untuk memastikan struktur keempat dataset sesuai dengan kebutuhan
for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(f"DATASET: {name}")
    print("=" * 60)

    # 1. Struktur kolom
    structure = pd.DataFrame({
        "column_name": df.columns,
        "data_type": df.dtypes.astype(str).values,
        "null_count": df.isna().sum().values,
        "null_percentage": (
            df.isna().mean() * 100
        ).round(2).values
    })

    print("\n[1] STRUKTUR KOLOM")
    display(structure)

    # 2. Contoh data
    print("\n[2] SAMPLE DATA (3 BARIS PERTAMA)")
    display(df.head(3))

print("\nSTEP 02 SELESAI.")


DATASET: Budget Detail

[1] STRUKTUR KOLOM


,column_name,data_type,null_count,null_percentage
0,source_entity,string,0,0.00
1,biro_code,string,0,0.00
2,biro_name,object,0,0.00
3,row_type,object,0,0.00
4,funds_center_code,string,1037,13.82
5,funds_center_name,object,1037,13.82
6,commitment_item_code,string,0,0.00
7,commitment_item_name,object,0,0.00
8,budget_scope,object,0,0.00
9,consumable_budget,int64,0,0.00



[2] SAMPLE DATA (3 BARIS PERTAMA)


,source_entity,biro_code,biro_name,row_type,funds_center_code,funds_center_name,commitment_item_code,commitment_item_name,budget_scope,consumable_budget,consumed_budget,available_amount,current_budget,commitment_actuals,sisa,calculated_available,available_difference
0,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,<NA>,NaN,4121902006,PEND SEWA TRUCK,CI_ONLY,0,0,0,0,-84200000,0,0,0
1,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,<NA>,NaN,4121909900,JASA LAINNYA,CI_ONLY,0,0,0,0,15000000,0,0,0
2,KANTOR PUSAT SILOG,001,GENERAL,DETAIL,<NA>,NaN,6214000000,Biaya BBM kendaraan dinas,CI_ONLY,48618900,48612552,6348,233421610,48612552,184802710,6348,0



DATASET: Master CI

[1] STRUKTUR KOLOM


...


DATASET: Master Fund

[1] STRUKTUR KOLOM


,column_name,data_type,null_count,null_percentage
0,from_profit_center,string,0,0.0
1,to_profit_center,string,0,0.0
2,target_fund_code,string,0,0.0



[2] SAMPLE DATA (3 BARIS PERTAMA)


,from_profit_center,to_profit_center,target_fund_code
0,S010204,S010204,S000537
1,S020101,S020101,S000534
2,S020104,S020104,S000537



STEP 02 SELESAI.


## STEP 03 — Cek Budget Negatif

**Tujuan:** Hitung baris yang memiliki nilai negatif pada kolom anggaran.

In [3]:
# step 03 - budget quality profiling
# untuk memeriksa nilai negatif pada 4 kolom utama
# current, consumable, consumed, available
budget_columns = [
    "current_budget",
    "consumable_budget",
    "consumed_budget",
    "available_amount"
]

# 1. Pastikan kolom budget bertipe numerik
for col in budget_columns:
    df_budget[col] = pd.to_numeric(
        df_budget[col],
        errors="coerce"
    )

# 2. Identifikasi nilai negatif
negative_mask = df_budget[budget_columns] < 0

# 3. Hitung nilai negatif per kolom
negative_summary = pd.DataFrame({
    "column_name": budget_columns,
    "negative_count": negative_mask.sum().values,
    "negative_percentage": (
        negative_mask.mean() * 100
    ).round(2).values
})

# 4. Identifikasi baris dengan minimal satu nilai negatif
has_negative = negative_mask.any(axis=1)

total_rows = len(df_budget)
total_negative_rows = has_negative.sum()
total_non_negative_rows = (~has_negative).sum()

# 5. Tampilkan hasil
print("=" * 55)
print("STEP 03 - NEGATIVE BUDGET PROFILING")
print("=" * 55)

print(f"\nTotal baris budget        : {total_rows:,}")
print(f"Baris dengan nilai negatif: {total_negative_rows:,}")
print(f"Baris tanpa nilai negatif : {total_non_negative_rows:,}")

print("\n[1] RINGKASAN NILAI NEGATIF PER KOLOM")
display(negative_summary)

print("\n[2] NILAI NEGATIF BERDASARKAN BUDGET SCOPE")

scope_negative = (
    df_budget.assign(has_negative=has_negative)
    .groupby("budget_scope", dropna=False)["has_negative"]
    .agg(
        total_rows="size",
        negative_rows="sum"
    )
    .reset_index()
)

scope_negative["non_negative_rows"] = (
    scope_negative["total_rows"]
    - scope_negative["negative_rows"]
)

display(scope_negative)

print("\nSTEP 03 SELESAI.")

STEP 03 - NEGATIVE BUDGET PROFILING

Total baris budget        : 7,504
Baris dengan nilai negatif: 62
Baris tanpa nilai negatif : 7,442

[1] RINGKASAN NILAI NEGATIF PER KOLOM


,column_name,negative_count,negative_percentage
0,current_budget,20,0.27
1,consumable_budget,0,0.00
2,consumed_budget,42,0.56
3,available_amount,1,0.01



[2] NILAI NEGATIF BERDASARKAN BUDGET SCOPE


,budget_scope,total_rows,negative_rows,non_negative_rows
0,CI_ONLY,1037,58,979
1,FC_CI,6467,4,6463



STEP 03 SELESAI.


## STEP 04 — Cek Perhitungan Budget

**Tujuan:** Bandingkan nilai Available dengan Consumable dikurangi Consumed. Cek juga Approval Hold.

In [4]:
# step 04 - validasi konsistensi nominal budget
# untuk mengetahui apakah nominal budget dalam budget detail
# available amount = consumable - consumed
# approval hold amount = current - consumable
# dengan syarat:
# approval hold = current budget >= consumable budget
# consumed budget = consumed budget <= consumable budget
# current budget = current budget >= consumable budget
import numpy as np

# 1. Buat dataframe untuk profiling
df_consistency = df_budget.copy()

# 2. Hitung nilai berdasarkan rumus sistem
df_consistency["expected_available"] = (
    df_consistency["consumable_budget"]
    - df_consistency["consumed_budget"]
)

df_consistency["approval_hold"] = (
    df_consistency["current_budget"]
    - df_consistency["consumable_budget"]
)

# 3. Validasi nilai budget
# Toleransi Rp1 untuk selisih pembulatan
tolerance = 1

complete_amounts = df_consistency[
    budget_columns
].notna().all(axis=1)

df_consistency["available_consistent"] = (
    np.isclose(
        df_consistency["available_amount"],
        df_consistency["expected_available"],
        atol=tolerance,
        rtol=0
    )
    & complete_amounts
)

df_consistency["approval_hold_valid"] = (
    (df_consistency["approval_hold"] >= -tolerance)
    & complete_amounts
)

df_consistency["consumed_valid"] = (
    (
        df_consistency["consumed_budget"]
        <= df_consistency["consumable_budget"] + tolerance
    )
    & complete_amounts
)

# 4. Identifikasi baris yang lolos semua pemeriksaan
validation_columns = [
    "available_consistent",
    "approval_hold_valid",
    "consumed_valid"
]

df_consistency["amount_consistent"] = (
    df_consistency[validation_columns].all(axis=1)
)

# 5. Ringkasan hasil
consistency_summary = pd.DataFrame({
    "validation": validation_columns,
    "valid_rows": [
        df_consistency[col].sum()
        for col in validation_columns
    ],
    "invalid_rows": [
        (~df_consistency[col]).sum()
        for col in validation_columns
    ]
})

# 6. Tampilkan output
print("=" * 60)
print("STEP 04 - BUDGET AMOUNT CONSISTENCY")
print("=" * 60)

print(f"\nTotal baris               : {len(df_consistency):,}")
print(f"Nominal lengkap           : {complete_amounts.sum():,}")
print(f"Nominal kosong/tidak valid: {(~complete_amounts).sum():,}")
print(f"Seluruh pemeriksaan lolos : {df_consistency['amount_consistent'].sum():,}")
print(f"Perlu review              : {(~df_consistency['amount_consistent']).sum():,}")

print("\n[1] HASIL VALIDASI PER KONDISI")
display(consistency_summary)

print("\n[2] CONTOH DATA TIDAK KONSISTEN (10 BARIS)")
display(
    df_consistency.loc[
        ~df_consistency["amount_consistent"],
        [
            "source_entity",
            "biro_code",
            "budget_scope",
            "funds_center_code",
            "commitment_item_code",
            "current_budget",
            "consumable_budget",
            "consumed_budget",
            "available_amount",
            "expected_available",
            "approval_hold"
        ]
    ].head(10)
)

print("\nSTEP 05 SELESAI.")

STEP 04 - BUDGET AMOUNT CONSISTENCY

Total baris               : 7,504
Nominal lengkap           : 7,504
Nominal kosong/tidak valid: 0
Seluruh pemeriksaan lolos : 7,481
Perlu review              : 23

[1] HASIL VALIDASI PER KONDISI


,validation,valid_rows,invalid_rows
0,available_consistent,7503,1
1,approval_hold_valid,7483,21
2,consumed_valid,7503,1



[2] CONTOH DATA TIDAK KONSISTEN (10 BARIS)


,source_entity,biro_code,budget_scope,funds_center_code,commitment_item_code,current_budget,consumable_budget,consumed_budget,available_amount,expected_available,approval_hold
22,KANTOR PUSAT SILOG,001,CI_ONLY,<NA>,6422000000,-28700000,0,0,0,0,-28700000
83,KANTOR PUSAT SILOG,220,CI_ONLY,<NA>,6432000900,-6200000,0,0,0,0,-6200000
113,KANTOR PUSAT SILOG,230,CI_ONLY,<NA>,6710000000,92602994,92571817,87641011,2230806,4930806,31177
191,KANTOR PUSAT SILOG,270,CI_ONLY,<NA>,6771000600,-38550000,0,0,0,0,-38550000
231,KANTOR PUSAT SILOG,320,CI_ONLY,<NA>,6742000200,-56076000,0,0,0,0,-56076000
235,KANTOR PUSAT SILOG,330,CI_ONLY,<NA>,6541000400,-617703515,0,0,0,0,-617703515
236,KANTOR PUSAT SILOG,330,CI_ONLY,<NA>,6541000409,-800000,0,0,0,0,-800000
238,KANTOR PUSAT SILOG,330,CI_ONLY,<NA>,6722004200,-8800000,0,0,0,0,-8800000
243,KANTOR PUSAT SILOG,330,CI_ONLY,<NA>,6784000100,-21000,0,0,0,0,-21000
270,KANTOR PUSAT SILOG,340,CI_ONLY,<NA>,6331000200,-108000,0,0,0,0,-108000



STEP 05 SELESAI.


## STEP 05 — Cek Budget Scope

**Tujuan:** Pastikan data `FC_CI` punya FC dan CI, sedangkan `CI_ONLY` punya CI.

In [5]:
# step 05 - budget scope validation
# untuk memastikan fc_ci dan ci_only sesuai dgn kebutuhan pencarian sumber budget
df_scope = df_budget.copy()

# 1. Normalisasi budget scope
df_scope["budget_scope_clean"] = (
    df_scope["budget_scope"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 2. Periksa kelengkapan kode
fc_present = (
    df_scope["funds_center_code"]
    .astype("string")
    .str.strip()
    .fillna("")
    .ne("")
)

ci_present = (
    df_scope["commitment_item_code"]
    .astype("string")
    .str.strip()
    .fillna("")
    .ne("")
)

# 3. Validasi berdasarkan budget scope
df_scope["scope_valid"] = (
    (
        (df_scope["budget_scope_clean"] == "FC_CI")
        & fc_present
        & ci_present
    )
    |
    (
        (df_scope["budget_scope_clean"] == "CI_ONLY")
        & ci_present
    )
)

# 4. Ringkasan per scope
scope_summary = (
    df_scope
    .groupby("budget_scope_clean", dropna=False)
    .agg(
        total_rows=("scope_valid", "size"),
        valid_rows=("scope_valid", "sum")
    )
    .reset_index()
)

scope_summary["invalid_rows"] = (
    scope_summary["total_rows"]
    - scope_summary["valid_rows"]
)

scope_summary["valid_percentage"] = (
    scope_summary["valid_rows"]
    / scope_summary["total_rows"] * 100
).round(2)

# 5. Tampilkan hasil
print("=" * 60)
print("STEP 05 - BUDGET SCOPE VALIDATION")
print("=" * 60)

display(scope_summary)

print("\nTotal scope valid:", df_scope["scope_valid"].sum())
print("Total scope invalid:", (~df_scope["scope_valid"]).sum())

print("\n[CONTOH DATA SCOPE INVALID]")

display(
    df_scope.loc[
        ~df_scope["scope_valid"],
        [
            "source_entity",
            "biro_code",
            "budget_scope",
            "funds_center_code",
            "commitment_item_code"
        ]
    ].head(10)
)

STEP 05 - BUDGET SCOPE VALIDATION


,budget_scope_clean,total_rows,valid_rows,invalid_rows,valid_percentage
0,CI_ONLY,1037,1037,0,100.0
1,FC_CI,6467,6467,0,100.0



Total scope valid: 7504
Total scope invalid: 0

[CONTOH DATA SCOPE INVALID]


,source_entity,biro_code,budget_scope,funds_center_code,commitment_item_code


## STEP 06 — Cek Data Master

**Tujuan:** Periksa kolom, data kosong, dan duplikat di Master CI, Master FC, dan Master Fund.

In [6]:
# step 06 - periksa struktur dan kualitas master ci, fc, dan fund
# untuk memastikan nama kolom master sesuai rancangan
# untuk hitung nilai kosong pada mapping
# untuk hitung jumlah duplikasi sama persis
# untuk jumlah kode target unik
master_datasets = {
    "Master CI": df_master_ci,
    "Master FC": df_master_fc,
    "Master Fund": df_master_fund
}

# 1. Daftar kolom target yang diharapkan
expected_columns = {
    "Master CI": [
        "From G/L Account",
        "To G/L Account",
        "Target Commitment item"
    ],
    "Master FC": [
        "From Cost Center",
        "To Cost Center",
        "Target Funds Center"
    ],
    "Master Fund": [
        "from_profit_center",
        "to_profit_center",
        "target_fund_code"
    ]
}

print("=" * 60)
print("STEP 06 - MASTER DATA PROFILING")
print("=" * 60)

for name, df in master_datasets.items():

    print(f"\n{'=' * 50}")
    print(f"DATASET: {name}")
    print("=" * 50)

    # 2. Nama kolom aktual
    actual_columns = df.columns.tolist()

    print("\n[1] NAMA KOLOM AKTUAL")
    for col in actual_columns:
        print(f"- {repr(col)}")

    # 3. Pemeriksaan kolom wajib
    # Abaikan spasi awal/akhir dan perbedaan kapital
    normalized_columns = {
        str(col).strip().lower(): col
        for col in actual_columns
    }

    missing_columns = [
        col
        for col in expected_columns[name]
        if col.strip().lower() not in normalized_columns
    ]

    if missing_columns:
        print("\n[PERINGATAN] Kolom belum ditemukan:")
        for col in missing_columns:
            print(f"- {col}")
    else:
        print("\nSemua kolom yang diharapkan ditemukan.")

    # 4. Ringkasan kualitas
    print("\n[2] RINGKASAN KUALITAS DATA")

    print(f"Total baris          : {len(df):,}")
    print(f"Total kolom          : {len(df.columns)}")
    print(f"Duplikasi baris      : {df.duplicated().sum():,}")
    print(f"Baris dengan null    : {df.isna().any(axis=1).sum():,}")

    # 5. Missing value per kolom
    print("\n[3] MISSING VALUE PER KOLOM")

    null_summary = pd.DataFrame({
        "column": df.columns,
        "null_count": df.isna().sum().values,
        "null_percentage": (
            df.isna().mean() * 100
        ).round(2).values
    })

    display(null_summary)

    # 6. Jumlah target unik
    if not missing_columns:
        target_col = normalized_columns[
            expected_columns[name][2].strip().lower()
        ]

        print(
            f"\nJumlah target unik: "
            f"{df[target_col].nunique(dropna=True):,}"
        )

print("\nSTEP 07 SELESAI.")

STEP 06 - MASTER DATA PROFILING

DATASET: Master CI

[1] NAMA KOLOM AKTUAL
- 'From G/L Account'
- 'To G/L Account'
- 'Target Commitment item'

Semua kolom yang diharapkan ditemukan.

[2] RINGKASAN KUALITAS DATA
Total baris          : 1,499
Total kolom          : 3
Duplikasi baris      : 0
Baris dengan null    : 0

[3] MISSING VALUE PER KOLOM


,column,null_count,null_percentage
0,From G/L Account,0,0.0
1,To G/L Account,0,0.0
2,Target Commitment item,0,0.0



Jumlah target unik: 1,449

DATASET: Master FC

[1] NAMA KOLOM AKTUAL
- 'From Cost Center'
- 'To Cost Center'
- 'Target Funds Center'

Semua kolom yang diharapkan ditemukan.

[2] RINGKASAN KUALITAS DATA
Total baris          : 1,776
Total kolom          : 3
Duplikasi baris      : 0
Baris dengan null    : 0

[3] MISSING VALUE PER KOLOM


,column,null_count,null_percentage
0,From Cost Center,0,0.0
1,To Cost Center,0,0.0
2,Target Funds Center,0,0.0



Jumlah target unik: 1,652

DATASET: Master Fund

[1] NAMA KOLOM AKTUAL
- 'from_profit_center'
- 'to_profit_center'
- 'target_fund_code'

Semua kolom yang diharapkan ditemukan.

[2] RINGKASAN KUALITAS DATA
Total baris          : 334
Total kolom          : 3
Duplikasi baris      : 0
Baris dengan null    : 0

[3] MISSING VALUE PER KOLOM


,column,null_count,null_percentage
0,from_profit_center,0,0.0
1,to_profit_center,0,0.0
2,target_fund_code,0,0.0



Jumlah target unik: 112

STEP 07 SELESAI.


## STEP 07 — Cocokkan Commitment Item

**Tujuan:** Cek apakah kode CI di budget ada di Target Commitment Item pada Master CI.

In [7]:
# step 07 - commitment item cross data validation
# untuk mengetahui jumlah data budget yang commitment itemnya ditemukan dalam master ci
# dan jumlah yang tidak ditemukan
# 1. Buat salinan untuk validasi
df_ci_validation = df_budget.copy()

# 2. Normalisasi kode Commitment Item
df_ci_validation["ci_code_clean"] = (
    df_ci_validation["commitment_item_code"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

master_ci_targets = (
    df_master_ci["Target Commitment item"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .dropna()
)

# 3. Buat daftar Target CI yang valid
valid_ci_codes = set(
    master_ci_targets[
        master_ci_targets.ne("")
    ]
)

# 4. Cek keberadaan CI budget pada Master CI
df_ci_validation["ci_found"] = (
    df_ci_validation["ci_code_clean"]
    .isin(valid_ci_codes)
)

# 5. Hitung ringkasan
total_rows = len(df_ci_validation)
found_rows = int(df_ci_validation["ci_found"].sum())
not_found_rows = total_rows - found_rows

print("=" * 60)
print("STEP 07 - COMMITMENT ITEM VALIDATION")
print("=" * 60)

print(f"\nTotal baris budget : {total_rows:,}")
print(f"CI ditemukan       : {found_rows:,}")
print(f"CI tidak ditemukan : {not_found_rows:,}")

print(
    f"Persentase ditemukan: "
    f"{found_rows / total_rows * 100:.2f}%"
    if total_rows else "Dataset kosong"
)

# 6. Ringkasan berdasarkan Budget Scope
print("\n[1] VALIDASI CI BERDASARKAN BUDGET SCOPE")

ci_scope_summary = (
    df_ci_validation
    .groupby("budget_scope", dropna=False)["ci_found"]
    .agg(
        total_rows="size",
        ci_found="sum"
    )
    .reset_index()
)

ci_scope_summary["ci_not_found"] = (
    ci_scope_summary["total_rows"]
    - ci_scope_summary["ci_found"]
)

display(ci_scope_summary)

# 7. Daftar CI yang tidak ditemukan
print("\n[2] DAFTAR COMMITMENT ITEM TIDAK DITEMUKAN")

ci_not_found_summary = (
    df_ci_validation.loc[
        ~df_ci_validation["ci_found"]
    ]
    .groupby(
        ["ci_code_clean", "commitment_item_name"],
        dropna=False
    )
    .size()
    .reset_index(name="total_rows")
    .sort_values("total_rows", ascending=False)
)

display(ci_not_found_summary)

STEP 07 - COMMITMENT ITEM VALIDATION

Total baris budget : 7,504
CI ditemukan       : 5,861
CI tidak ditemukan : 1,643
Persentase ditemukan: 78.11%

[1] VALIDASI CI BERDASARKAN BUDGET SCOPE


,budget_scope,total_rows,ci_found,ci_not_found
0,CI_ONLY,1037,962,75
1,FC_CI,6467,4899,1568



[2] DAFTAR COMMITMENT ITEM TIDAK DITEMUKAN


,ci_code_clean,commitment_item_name,total_rows
14,67220047001,KEPERLUAN CLOSING,1566
12,6541000407,PEMAKAIAN CONSUMMABLE PARTS,33
25,6772000400,JASA TENAGA ADMINISTRASI,8
16,6742000100,PEMAKAIAN SUPPLAI KANTOR,6
20,6761000700,BIAYA KEUR TRUCK,3
15,6734000000,Biaya Laka & Tilang,3
4,5911000304,BIAYA ONGKOS ANGKUT TRUCK LUAR,2
5,5911000307,BIAYA SEWA ALAT BERAT,2
6,5911000310,BIAYA KULI PELABUHAN,2
22,67710005001,SEWA PERLENGKAPAN DAN PERALATAN MULTIYEARS,2


## STEP 08 — Gabungkan Hasil Validasi

**Tujuan:** Tandai data yang lolos: tidak negatif, CI ditemukan, scope benar, dan nominal konsisten.

In [8]:
# step 08 - combined data validation
# untuk mengetahui baris yang memenuhi kriteria
# tidak memiliki nilai negatif
# commitment item ditemukan pada master ci
# struktur budget_scope valid
# nominal budget konsisten dengan rumus sistem
# 1. Buat dataframe untuk hasil validasi gabungan
df_readiness = df_budget.copy()

# 2. Gabungkan status pemeriksaan sebelumnya
df_readiness["no_negative"] = ~(
    df_budget[budget_columns] < 0
).any(axis=1)

df_readiness["amount_complete"] = (
    df_budget[budget_columns].notna().all(axis=1)
)

df_readiness["ci_found"] = (
    df_ci_validation["ci_found"]
)

df_readiness["scope_valid"] = (
    df_scope["scope_valid"]
)

df_readiness["amount_consistent"] = (
    df_consistency["amount_consistent"]
)

# 3. Validasi berdasarkan kesepakatan user
df_readiness["meets_user_criteria"] = (
    df_readiness["no_negative"]
    & df_readiness["ci_found"]
)

# 4. Validasi kesiapan teknis pengembangan
df_readiness["dev_ready"] = (
    df_readiness["meets_user_criteria"]
    & df_readiness["amount_complete"]
    & df_readiness["scope_valid"]
    & df_readiness["amount_consistent"]
)

# 5. Ringkasan validasi
validation_summary = pd.DataFrame({
    "criteria": [
        "No Negative",
        "Amount Complete",
        "CI Found",
        "Scope Valid",
        "Amount Consistent",
        "Meets User Criteria",
        "Development Ready"
    ],
    "passed_rows": [
        df_readiness["no_negative"].sum(),
        df_readiness["amount_complete"].sum(),
        df_readiness["ci_found"].sum(),
        df_readiness["scope_valid"].sum(),
        df_readiness["amount_consistent"].sum(),
        df_readiness["meets_user_criteria"].sum(),
        df_readiness["dev_ready"].sum()
    ]
})

validation_summary["failed_rows"] = (
    len(df_readiness)
    - validation_summary["passed_rows"]
)

# 6. Tampilkan hasil
print("=" * 60)
print("STEP 08 - COMBINED DATA VALIDATION")
print("=" * 60)

display(validation_summary)

print("\n[1] FINAL READINESS SUMMARY")

print(f"Total Budget        : {len(df_readiness):,}")
print(
    f"Meets User Criteria : "
    f"{df_readiness['meets_user_criteria'].sum():,}"
)
print(
    f"Development Ready   : "
    f"{df_readiness['dev_ready'].sum():,}"
)
print(
    f"Not Development Ready: "
    f"{(~df_readiness['dev_ready']).sum():,}"
)

print("\n[2] DEVELOPMENT READY BY BUDGET SCOPE")

ready_scope_summary = (
    df_readiness
    .groupby("budget_scope", dropna=False)
    .agg(
        total_rows=("dev_ready", "size"),
        user_criteria_rows=("meets_user_criteria", "sum"),
        dev_ready_rows=("dev_ready", "sum")
    )
    .reset_index()
)

display(ready_scope_summary)

print("\nSTEP 09 SELESAI.")

STEP 08 - COMBINED DATA VALIDATION


,criteria,passed_rows,failed_rows
0,No Negative,7442,62
1,Amount Complete,7504,0
2,CI Found,5861,1643
3,Scope Valid,7504,0
4,Amount Consistent,7481,23
5,Meets User Criteria,5811,1693
6,Development Ready,5809,1695



[1] FINAL READINESS SUMMARY
Total Budget        : 7,504
Meets User Criteria : 5,811
Development Ready   : 5,809
Not Development Ready: 1,695

[2] DEVELOPMENT READY BY BUDGET SCOPE


,budget_scope,total_rows,user_criteria_rows,dev_ready_rows
0,CI_ONLY,1037,916,914
1,FC_CI,6467,4895,4895



STEP 09 SELESAI.


## STEP 09 — Lihat Sebaran Data Valid

**Tujuan:** Hitung jumlah Entity, Biro, Funds Center, CI, dan baris valid per Entity.

In [9]:
# step 09 - development ready data distribution
# untuk mengetahui dari 5.809 yang lolos validasi
# dicek apakah:
# jumlah entity yang terwakili
# jumlah biro yang terwakili
# jumlah funds center unik
# jumlah commitment item unik
# distribusi data berdasarkan entity dan budget scope
# 1. Ambil data yang lolos seluruh validasi awal
df_ready = df_readiness.loc[
    df_readiness["dev_ready"]
].copy()

# 2. Hitung keragaman data
total_entities = df_ready["source_entity"].nunique()

total_biros = (
    df_ready[["source_entity", "biro_code"]]
    .drop_duplicates()
    .shape[0]
)

total_funds_centers = (
    df_ready["funds_center_code"].nunique()
)

total_commitment_items = (
    df_ready["commitment_item_code"].nunique()
)

print("=" * 60)
print("STEP 09 - DEVELOPMENT READY DISTRIBUTION")
print("=" * 60)

print(f"\nTotal data ready        : {len(df_ready):,}")
print(f"Total Entity            : {total_entities:,}")
print(f"Total Biro              : {total_biros:,}")
print(f"Total Funds Center      : {total_funds_centers:,}")
print(f"Total Commitment Item   : {total_commitment_items:,}")

# 3. Distribusi berdasarkan Entity dan Budget Scope
print("\n[1] DISTRIBUSI ENTITY + BUDGET SCOPE")

entity_scope_summary = (
    df_ready
    .groupby(
        ["source_entity", "budget_scope"],
        dropna=False
    )
    .size()
    .unstack(fill_value=0)
)

entity_scope_summary["total_rows"] = (
    entity_scope_summary.sum(axis=1)
)

entity_scope_summary = (
    entity_scope_summary
    .sort_values("total_rows", ascending=False)
)

display(entity_scope_summary)

# 4. Distribusi jumlah Biro per Entity
print("\n[2] JUMLAH BIRO PER ENTITY")

biro_entity_summary = (
    df_ready
    .groupby("source_entity", dropna=False)
    .agg(
        total_rows=("biro_code", "size"),
        unique_biro=("biro_code", "nunique"),
        unique_fc=("funds_center_code", "nunique"),
        unique_ci=("commitment_item_code", "nunique")
    )
    .reset_index()
    .sort_values("total_rows", ascending=False)
)

display(biro_entity_summary)

print("\nSTEP 09 SELESAI.")

STEP 09 - DEVELOPMENT READY DISTRIBUTION

Total data ready        : 5,809
Total Entity            : 7
Total Biro              : 52
Total Funds Center      : 1,464
Total Commitment Item   : 150

[1] DISTRIBUSI ENTITY + BUDGET SCOPE


budget_scope,CI_ONLY,FC_CI,total_rows
source_entity,,,
LOGISTIK,155,4755,4910
KANTOR PUSAT SILOG,458,108,566
VUBA,85,0,85
BUBI,51,31,82
VULS,73,0,73
VUDS,71,0,71
IKN,21,1,22



[2] JUMLAH BIRO PER ENTITY


,source_entity,total_rows,unique_biro,unique_fc,unique_ci
3,LOGISTIK,4910,16,1464,79
2,KANTOR PUSAT SILOG,566,26,108,100
4,VUBA,85,2,0,84
0,BUBI,82,3,31,51
6,VULS,73,1,0,73
5,VUDS,71,3,0,70
1,IKN,22,1,1,22



STEP 09 SELESAI.


## STEP 10 — Cek Sumber Budget Ganda

**Tujuan:** Pastikan kombinasi identitas sumber budget tidak muncul lebih dari sekali.

In [10]:
# step 10 - budget uniqueness profiling
# untuk memeriksa apakah terdapat beberapa baris budget dengan kombinasi identitas sumber yang sama
df_uniqueness = df_ready.copy()

# 1. Pisahkan berdasarkan Budget Scope
df_fc_ci = df_uniqueness[
    df_uniqueness["budget_scope"] == "FC_CI"
].copy()

df_ci_only = df_uniqueness[
    df_uniqueness["budget_scope"] == "CI_ONLY"
].copy()

# 2. Tentukan kandidat kunci sumber budget
fc_ci_keys = [
    "source_entity",
    "biro_code",
    "funds_center_code",
    "commitment_item_code"
]

ci_only_keys = [
    "source_entity",
    "biro_code",
    "commitment_item_code"
]

# 3. Identifikasi duplikasi kombinasi
df_fc_ci["duplicate_source"] = (
    df_fc_ci.duplicated(
        subset=fc_ci_keys,
        keep=False
    )
)

df_ci_only["duplicate_source"] = (
    df_ci_only.duplicated(
        subset=ci_only_keys,
        keep=False
    )
)

# 4. Ringkasan duplikasi
uniqueness_summary = pd.DataFrame({
    "budget_scope": ["FC_CI", "CI_ONLY"],
    "total_rows": [
        len(df_fc_ci),
        len(df_ci_only)
    ],
    "duplicate_rows": [
        int(df_fc_ci["duplicate_source"].sum()),
        int(df_ci_only["duplicate_source"].sum())
    ],
    "duplicate_groups": [
        df_fc_ci.loc[
            df_fc_ci["duplicate_source"],
            fc_ci_keys
        ].drop_duplicates().shape[0],

        df_ci_only.loc[
            df_ci_only["duplicate_source"],
            ci_only_keys
        ].drop_duplicates().shape[0]
    ]
})

uniqueness_summary["unique_rows"] = (
    uniqueness_summary["total_rows"]
    - uniqueness_summary["duplicate_rows"]
)

# 5. Tampilkan hasil
print("=" * 60)
print("STEP 10 - BUDGET SOURCE UNIQUENESS")
print("=" * 60)

display(uniqueness_summary)

print("\n[1] CONTOH DUPLIKASI FC_CI")

display(
    df_fc_ci.loc[
        df_fc_ci["duplicate_source"],
        fc_ci_keys + budget_columns
    ].sort_values(fc_ci_keys).head(10)
)

print("\n[2] CONTOH DUPLIKASI CI_ONLY")

display(
    df_ci_only.loc[
        df_ci_only["duplicate_source"],
        ci_only_keys + budget_columns
    ].sort_values(ci_only_keys).head(10)
)

print("\nSTEP 10 SELESAI.")

STEP 10 - BUDGET SOURCE UNIQUENESS


,budget_scope,total_rows,duplicate_rows,duplicate_groups,unique_rows
0,FC_CI,4895,0,0,4895
1,CI_ONLY,914,0,0,914



[1] CONTOH DUPLIKASI FC_CI


,source_entity,biro_code,funds_center_code,commitment_item_code,current_budget,consumable_budget,consumed_budget,available_amount



[2] CONTOH DUPLIKASI CI_ONLY


,source_entity,biro_code,commitment_item_code,current_budget,consumable_budget,consumed_budget,available_amount



STEP 10 SELESAI.


**Hasil dari STEP 10:** tidak ditemukan duplikasi sumber budget pada kombinasi kunci yang diperiksa.

## STEP 11 — Cocokkan Funds Center

**Tujuan:** Cek FC pada `FC_CI` terhadap Target Funds Center di Master FC.

In [11]:
# step 11 - funds center cross-data validation
# untuk membandingkan funds center code dengan target funds center
# khusus fc_ci, kode funds center harus ditemukan pada master fc
# sementara ci_only, funds center boleh kosong karena tidak digunakan sebagai kunci pencarian budget
# 1. Buat salinan data development ready
df_fc_validation = df_ready.copy()

# 2. Normalisasi kode Funds Center pada budget
df_fc_validation["fc_code_clean"] = (
    df_fc_validation["funds_center_code"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

# 3. Normalisasi Target Funds Center Master FC
master_fc_targets = (
    df_master_fc["Target Funds Center"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .dropna()
)

valid_fc_codes = set(
    master_fc_targets[master_fc_targets.ne("")]
)

# 4. Validasi berdasarkan budget scope
df_fc_validation["fc_found"] = (
    df_fc_validation["fc_code_clean"]
    .isin(valid_fc_codes)
)

# CI_ONLY tidak membutuhkan FC untuk pencarian budget
df_fc_validation["fc_validation_pass"] = (
    (df_fc_validation["budget_scope"] == "CI_ONLY")
    |
    (
        (df_fc_validation["budget_scope"] == "FC_CI")
        & df_fc_validation["fc_found"]
    )
)

# 5. Ringkasan berdasarkan Budget Scope
fc_summary = (
    df_fc_validation
    .groupby("budget_scope", dropna=False)
    .agg(
        total_rows=("fc_found", "size"),
        fc_found=("fc_found", "sum"),
        validation_pass=("fc_validation_pass", "sum")
    )
    .reset_index()
)

fc_summary["fc_not_found"] = (
    fc_summary["total_rows"] - fc_summary["fc_found"]
)

fc_summary["validation_failed"] = (
    fc_summary["total_rows"]
    - fc_summary["validation_pass"]
)

print("=" * 60)
print("STEP 11 - FUNDS CENTER VALIDATION")
print("=" * 60)

display(fc_summary)

# 6. Daftar kode FC_CI yang tidak ditemukan
print("\n[1] FUNDS CENTER FC_CI TIDAK DITEMUKAN")

fc_not_found_summary = (
    df_fc_validation.loc[
        (df_fc_validation["budget_scope"] == "FC_CI")
        & (~df_fc_validation["fc_found"])
    ]
    .groupby(
        ["fc_code_clean", "funds_center_name"],
        dropna=False
    )
    .size()
    .reset_index(name="total_rows")
    .sort_values("total_rows", ascending=False)
)

display(fc_not_found_summary.head(30))

print("\n[2] FINAL SUMMARY")

print(
    "Total FC validation pass:",
    int(df_fc_validation["fc_validation_pass"].sum())
)

print(
    "Total FC validation failed:",
    int((~df_fc_validation["fc_validation_pass"]).sum())
)

print("\nSTEP 11 SELESAI.")

STEP 11 - FUNDS CENTER VALIDATION


,budget_scope,total_rows,fc_found,validation_pass,fc_not_found,validation_failed
0,CI_ONLY,914,0,914,914,0
1,FC_CI,4895,4895,4895,0,0



[1] FUNDS CENTER FC_CI TIDAK DITEMUKAN


,fc_code_clean,funds_center_name,total_rows



[2] FINAL SUMMARY
Total FC validation pass: 5809
Total FC validation failed: 0

STEP 11 SELESAI.


## STEP 12 — Kelompokkan Kondisi Budget

**Tujuan:** Kelompokkan budget yang masih tersedia, habis, atau masih punya Approval Hold.

In [12]:
# step 13 - budget scenario profiling
# untuk mengelompokkan 5.809 baris berdasarkan kondisi ketersediaan budget
# untuk mengetahui apakah data yang tersedia dapat mendukung beberapa skenario pengujian
import numpy as np

# 1. Gunakan data development ready
df_scenario = df_fc_validation.loc[
    df_fc_validation["fc_validation_pass"]
].copy()

# 2. Hitung Approval Hold
df_scenario["approval_hold"] = (
    df_scenario["current_budget"]
    - df_scenario["consumable_budget"]
)

# 3. Klasifikasi kondisi budget
conditions = [
    # Budget tersedia dan masih ada yang bisa dibuka
    (
        (df_scenario["available_amount"] > 0)
        & (df_scenario["approval_hold"] > 0)
    ),

    # Budget tersedia, tetapi tidak ada Approval Hold
    (
        (df_scenario["available_amount"] > 0)
        & (df_scenario["approval_hold"] <= 0)
    ),

    # Available habis, tetapi masih bisa Open Budget
    (
        (df_scenario["available_amount"] == 0)
        & (df_scenario["approval_hold"] > 0)
    ),

    # Available habis dan tidak ada Approval Hold
    (
        (df_scenario["available_amount"] == 0)
        & (df_scenario["approval_hold"] <= 0)
    )
]

labels = [
    "AVAILABLE_WITH_HOLD",
    "AVAILABLE_NO_HOLD",
    "ZERO_AVAILABLE_WITH_HOLD",
    "ZERO_AVAILABLE_NO_HOLD"
]

df_scenario["budget_condition"] = np.select(
    conditions,
    labels,
    default="OTHER"
)

# 4. Ringkasan kondisi
scenario_summary = (
    df_scenario
    .groupby(
        ["budget_scope", "budget_condition"],
        dropna=False
    )
    .agg(
        total_rows=("budget_condition", "size"),
        total_available=("available_amount", "sum"),
        total_approval_hold=("approval_hold", "sum")
    )
    .reset_index()
)

# 5. Tampilkan hasil
print("=" * 60)
print("STEP 12 - BUDGET SCENARIO PROFILING")
print("=" * 60)

print("\n[1] DISTRIBUSI KONDISI BUDGET")
display(scenario_summary)

print("\n[2] TOTAL BERDASARKAN KONDISI")

condition_summary = (
    df_scenario["budget_condition"]
    .value_counts()
    .reset_index()
)

condition_summary.columns = [
    "budget_condition",
    "total_rows"
]

display(condition_summary)

print("\n[3] TOTAL DATA")
print(f"Total baris dianalisis: {len(df_scenario):,}")

print("\nSTEP 12 SELESAI.")

STEP 12 - BUDGET SCENARIO PROFILING

[1] DISTRIBUSI KONDISI BUDGET


,budget_scope,budget_condition,total_rows,total_available,total_approval_hold
0,CI_ONLY,AVAILABLE_NO_HOLD,72,4360317489,0
1,CI_ONLY,AVAILABLE_WITH_HOLD,182,1269239170975,194395271375
2,CI_ONLY,ZERO_AVAILABLE_NO_HOLD,595,0,0
3,CI_ONLY,ZERO_AVAILABLE_WITH_HOLD,65,0,6614062284
4,FC_CI,AVAILABLE_NO_HOLD,1131,6259287873,0
5,FC_CI,AVAILABLE_WITH_HOLD,281,1876701932,2218973119
6,FC_CI,ZERO_AVAILABLE_NO_HOLD,3483,0,0



[2] TOTAL BERDASARKAN KONDISI


,budget_condition,total_rows
0,ZERO_AVAILABLE_NO_HOLD,4078
1,AVAILABLE_NO_HOLD,1203
2,AVAILABLE_WITH_HOLD,463
3,ZERO_AVAILABLE_WITH_HOLD,65



[3] TOTAL DATA
Total baris dianalisis: 5,809

STEP 12 SELESAI.


**Arti kelompok:** Available/Zero menunjukkan ketersediaan budget; With/No Hold menunjukkan ada atau tidaknya dana yang masih bisa dibuka.

## STEP 13 — Cari calon realokasi

**Tujuan:** Cari pasangan budget target dan donor berdasarkan aturan `FC_CI` atau `CI_ONLY`.

In [13]:
# step 14 - realokasi budget candidate profiling
# untuk mencari kandidat pasangan sumber dan target realokasi berdasarkan aturan sistem
# syarat:
# fc_ci = funds center dan commitment item harus sama
# ci_only = commitment item harus sama
# 1. Gunakan data scenario
df_reallocation = df_scenario.copy()

# 2. Normalisasi kode untuk pencocokan
for col in [
    "funds_center_code",
    "commitment_item_code"
]:
    df_reallocation[col] = (
        df_reallocation[col]
        .astype("string")
        .str.strip()
    )

# 3. Identitas sumber budget
identity_cols = ["source_entity", "biro_code"]

# 4. Analisis per budget scope
results = []
candidate_examples = []

for scope, match_keys in {
    "FC_CI": [
        "funds_center_code",
        "commitment_item_code"
    ],
    "CI_ONLY": [
        "commitment_item_code"
    ]
}.items():

    data = df_reallocation[
        df_reallocation["budget_scope"] == scope
    ].copy()

    # Target: anggaran habis dan tidak ada hold
    targets = data[
        (data["available_amount"] == 0)
        & (data["approval_hold"] == 0)
    ].copy()

    # Donor: memiliki anggaran tersedia
    donors = data[
        data["available_amount"] > 0
    ].copy()

    # Pasangkan berdasarkan aturan scope
    pairs = targets.merge(
        donors,
        on=match_keys,
        suffixes=("_target", "_donor")
    )

    # Donor harus berasal dari Entity atau Biro lain
    different_source = (
        (pairs["source_entity_target"]
         != pairs["source_entity_donor"])
        |
        (pairs["biro_code_target"]
         != pairs["biro_code_donor"])
    )

    pairs = pairs[different_source].copy()

    # Jumlah target unik yang punya calon donor
    matched_target_count = (
        pairs["index_target"].nunique()
        if "index_target" in pairs.columns
        else 0
    )

    # Gunakan index asli sebagai identifier profiling
    # Tidak menggunakan Budget_ID database
    if len(pairs) > 0:
        pairs["target_key"] = (
            pairs["source_entity_target"].astype(str)
            + "|"
            + pairs["biro_code_target"].astype(str)
            + "|"
            + pairs["commitment_item_code"].astype(str)
        )

        if scope == "FC_CI":
            pairs["target_key"] += (
                "|" + pairs["funds_center_code"].astype(str)
            )

        matched_target_count = pairs[
            "target_key"
        ].nunique()

        examples = pairs[
            [
                "source_entity_target",
                "biro_code_target",
                "source_entity_donor",
                "biro_code_donor",
                "commitment_item_code",
                "available_amount_donor"
            ] + (
                ["funds_center_code"]
                if scope == "FC_CI" else []
            )
        ].head(5).copy()

        examples.insert(0, "budget_scope", scope)
        candidate_examples.append(examples)

    results.append({
        "budget_scope": scope,
        "total_targets": len(targets),
        "total_donors": len(donors),
        "targets_with_donor": matched_target_count,
        "candidate_pairs": len(pairs)
    })

# 5. Tampilkan ringkasan
reallocation_summary = pd.DataFrame(results)

print("=" * 60)
print("STEP 13 - REALLOCATION CANDIDATE PROFILING")
print("=" * 60)

display(reallocation_summary)

print("\n[1] CONTOH PASANGAN REALOKASI")

if candidate_examples:
    display(
        pd.concat(candidate_examples, ignore_index=True)
    )
else:
    print("Tidak ditemukan pasangan kandidat realokasi.")

print("\nSTEP 13 SELESAI.")

STEP 13 - REALLOCATION CANDIDATE PROFILING


,budget_scope,total_targets,total_donors,targets_with_donor,candidate_pairs
0,FC_CI,3483,1412,1,1
1,CI_ONLY,595,254,23,160



[1] CONTOH PASANGAN REALOKASI


,budget_scope,source_entity_target,biro_code_target,source_entity_donor,biro_code_donor,commitment_item_code,available_amount_donor,funds_center_code
0,FC_CI,LOGISTIK,460,IKN,33Z,6761000000,2000000,S004BL0324
1,CI_ONLY,KANTOR PUSAT SILOG,210,KANTOR PUSAT SILOG,001,6742000200,87150,<NA>
2,CI_ONLY,KANTOR PUSAT SILOG,210,KANTOR PUSAT SILOG,230,6742000200,23454,<NA>
3,CI_ONLY,KANTOR PUSAT SILOG,210,KANTOR PUSAT SILOG,340,6742000200,35000,<NA>
4,CI_ONLY,KANTOR PUSAT SILOG,210,KANTOR PUSAT SILOG,410,6742000200,19100,<NA>
5,CI_ONLY,KANTOR PUSAT SILOG,210,KANTOR PUSAT SILOG,440,6742000200,12114027,<NA>



STEP 13 SELESAI.


**Cara baca:** `total_targets` = target yang habis; `total_donors` = kandidat donor; `candidate_pairs` = pasangan yang cocok.

## STEP 14 — Lihat Detail Calon Realokasi

**Tujuan:** Tampilkan pasangan target–donor yang bisa dipakai sebagai contoh pengujian.

In [14]:
# step 14 - realokasi candidate detail validation
# untuk:
# identitas target dan donor
# nominal Available dan Approval Hold
# perbedaan Entity dan Biro
# jumlah donor yang tersedia untuk setiap target
reallocation_details = []

for scope, match_keys in {
    "FC_CI": [
        "funds_center_code",
        "commitment_item_code"
    ],
    "CI_ONLY": [
        "commitment_item_code"
    ]
}.items():

    data = df_reallocation[
        df_reallocation["budget_scope"] == scope
    ].copy()

    # Simpan identitas baris asli
    data["source_row_id"] = data.index

    targets = data[
        (data["available_amount"] == 0)
        & (data["approval_hold"] == 0)
    ].copy()

    donors = data[
        data["available_amount"] > 0
    ].copy()

    pairs = targets.merge(
        donors,
        on=match_keys,
        suffixes=("_target", "_donor")
    )

    # Harus berasal dari biro/entity berbeda
    different_source = (
        (pairs["source_entity_target"]
         != pairs["source_entity_donor"])
        |
        (pairs["biro_code_target"]
         != pairs["biro_code_donor"])
    )

    pairs = pairs[different_source].copy()

    if pairs.empty:
        continue

    pairs["budget_scope"] = scope

    reallocation_details.append(pairs)

# Gabungkan seluruh pasangan kandidat
if reallocation_details:

    df_reallocation_pairs = pd.concat(
        reallocation_details,
        ignore_index=True
    )

    # Hitung banyaknya donor untuk tiap target
    donor_count = (
        df_reallocation_pairs
        .groupby(
            ["budget_scope", "source_row_id_target"]
        )["source_row_id_donor"]
        .nunique()
        .reset_index(name="donor_count")
    )

    df_reallocation_pairs = (
        df_reallocation_pairs.merge(
            donor_count,
            on=["budget_scope", "source_row_id_target"],
            how="left"
        )
    )

    print("=" * 60)
    print("STEP 14 - REALLOCATION CANDIDATE DETAILS")
    print("=" * 60)

    print(f"\nTotal kandidat pasangan: {len(df_reallocation_pairs):,}")

    print("\n[1] DETAIL KANDIDAT REALOKASI FC_CI")

    display(
        df_reallocation_pairs.loc[
            df_reallocation_pairs["budget_scope"] == "FC_CI",
            [
                "source_row_id_target",
                "source_entity_target",
                "biro_code_target",
                "source_row_id_donor",
                "source_entity_donor",
                "biro_code_donor",
                "funds_center_code",
                "commitment_item_code",
                "available_amount_donor",
                "donor_count"
            ]
        ]
    )

    print("\n[2] TOP 10 KANDIDAT REALOKASI CI_ONLY")

    display(
        df_reallocation_pairs.loc[
            df_reallocation_pairs["budget_scope"] == "CI_ONLY",
            [
                "source_row_id_target",
                "source_entity_target",
                "biro_code_target",
                "source_row_id_donor",
                "source_entity_donor",
                "biro_code_donor",
                "commitment_item_code",
                "available_amount_donor",
                "donor_count"
            ]
        ]
        .sort_values(
            ["available_amount_donor"],
            ascending=False
        )
        .head(10)
    )

else:
    print("Tidak ditemukan kandidat realokasi.")

print("\nSTEP 14 SELESAI.")

STEP 14 - REALLOCATION CANDIDATE DETAILS

Total kandidat pasangan: 161

[1] DETAIL KANDIDAT REALOKASI FC_CI


,source_row_id_target,source_entity_target,biro_code_target,source_row_id_donor,source_entity_donor,biro_code_donor,funds_center_code,commitment_item_code,available_amount_donor,donor_count
0,1700,LOGISTIK,460,7244,IKN,33Z,S004BL0324,6761000000,2000000,1



[2] TOP 10 KANDIDAT REALOKASI CI_ONLY


,source_row_id_target,source_entity_target,biro_code_target,source_row_id_donor,source_entity_donor,biro_code_donor,commitment_item_code,available_amount_donor,donor_count
49,81,KANTOR PUSAT SILOG,220,7459,VULS,710,6422000000,522100900,10
59,311,KANTOR PUSAT SILOG,410,7459,VULS,710,6422000000,522100900,10
42,81,KANTOR PUSAT SILOG,220,185,KANTOR PUSAT SILOG,270,6422000000,391486745,10
52,311,KANTOR PUSAT SILOG,410,185,KANTOR PUSAT SILOG,270,6422000000,391486745,10
93,186,KANTOR PUSAT SILOG,270,672,LOGISTIK,330,6432000900,309230000,4
89,112,KANTOR PUSAT SILOG,230,672,LOGISTIK,330,6432000900,309230000,4
53,311,KANTOR PUSAT SILOG,410,213,KANTOR PUSAT SILOG,310,6422000000,222966061,10
43,81,KANTOR PUSAT SILOG,220,213,KANTOR PUSAT SILOG,310,6422000000,222966061,10
57,311,KANTOR PUSAT SILOG,410,670,LOGISTIK,330,6422000000,219193932,10
47,81,KANTOR PUSAT SILOG,220,670,LOGISTIK,330,6422000000,219193932,10



STEP 14 SELESAI.


## STEP 15 — Simpan Budget Valid

**Tujuan:** Simpan baris yang lolos validasi ke `data/ready/budget_valid.csv`.

In [15]:
# step 15 - finalisasi dataset budget
# untuk membuat dataset budget yang siap database
# mengambil 5.809 baris valid mempertahankan kolom yang dibutuhkan tabel budget_master dan menyimpan sebagai csv
from pathlib import Path

# 1. Buat folder output
OUTPUT_DIR = Path("data/ready")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Ambil seluruh data Development Ready
df_budget_final = df_readiness.loc[
    df_readiness["dev_ready"]
].copy()

# 3. Kolom yang dibutuhkan untuk persiapan database
selected_columns = [
    "source_entity",
    "biro_code",
    "biro_name",
    "funds_center_code",
    "funds_center_name",
    "commitment_item_code",
    "commitment_item_name",
    "budget_scope",
    "current_budget",
    "consumable_budget",
    "consumed_budget",
    "available_amount"
]

df_budget_final = df_budget_final[selected_columns]

# 4. Simpan CSV
output_file = OUTPUT_DIR / "budget_valid.csv"

df_budget_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

# 5. Ringkasan
print("=" * 55)
print("STEP 16 - FINAL DATASET")
print("=" * 55)

print(f"Total baris : {len(df_budget_final):,}")
print(f"Total kolom : {len(df_budget_final.columns)}")
print(f"File output : {output_file}")

print("\nDATASET BERHASIL DISIAPKAN.")

STEP 16 - FINAL DATASET
Total baris : 5,809
Total kolom : 12
File output : data/ready/budget_valid.csv

DATASET BERHASIL DISIAPKAN.


## STEP 16 — Buat CSV Tabel Supabase

**Tujuan:** Bentuk tabel master, budget, dan template transaksi sesuai ERD. Simpan hasilnya ke `data/supabase_import/`.

In [16]:
# step 16 - generate csv smeua tabel database
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data/supabase_import")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def clean_code(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace("", pd.NA)
    )

def create_ids(df, id_column):
    df = df.reset_index(drop=True).copy()
    df.insert(0, id_column, range(1, len(df) + 1))
    return df

# Gunakan hasil profiling sebelumnya
budget = df_readiness.loc[
    df_readiness["dev_ready"]
].copy()

ci_source = df_master_ci.copy()
fc_source = df_master_fc.copy()
fund_source = df_master_fund.copy()

# Bersihkan nama kolom master
for df in [ci_source, fc_source, fund_source]:
    df.columns = df.columns.str.strip()

# Normalisasi kode budget
for col in [
    "source_entity",
    "biro_code",
    "funds_center_code",
    "commitment_item_code",
    "budget_scope"
]:
    budget[col] = clean_code(budget[col])

# Normalisasi nama referensi
for col in [
    "biro_name",
    "funds_center_name",
    "commitment_item_name"
]:
    budget[col] = budget[col].astype("string").str.strip()
    budget[col] = budget[col].replace("", pd.NA)

# Normalisasi seluruh kode mapping
ci_source["from_code"] = clean_code(
    ci_source["From G/L Account"]
)
ci_source["to_code"] = clean_code(
    ci_source["To G/L Account"]
)
ci_source["target_code"] = clean_code(
    ci_source["Target Commitment item"]
)

fc_source["from_code"] = clean_code(
    fc_source["From Cost Center"]
)
fc_source["to_code"] = clean_code(
    fc_source["To Cost Center"]
)
fc_source["target_code"] = clean_code(
    fc_source["Target Funds Center"]
)

fund_source["from_code"] = clean_code(
    fund_source["from_profit_center"]
)
fund_source["to_code"] = clean_code(
    fund_source["to_profit_center"]
)
fund_source["target_code"] = clean_code(
    fund_source["target_fund_code"]
)

assert len(budget) == 5809, "Jumlah budget berubah!"
assert budget["source_entity"].notna().all()
assert budget["biro_code"].notna().all()
assert budget["commitment_item_code"].notna().all()
assert budget["budget_scope"].isin(["FC_CI", "CI_ONLY"]).all()

for source in [ci_source, fc_source, fund_source]:
    assert source[
        ["from_code", "to_code", "target_code"]
    ].notna().all().all(), "Mapping memiliki kode kosong"


# 2. UNIT_ENTITY
unit_entity = pd.DataFrame({
    "unit_entity": sorted(
        budget["source_entity"].dropna().unique()
    )
})

unit_entity = create_ids(unit_entity, "entity_id")

entity_lookup = dict(zip(
    unit_entity["unit_entity"],
    unit_entity["entity_id"]
))


# 3. BIRO
biro = (
    budget[
        ["source_entity", "biro_code", "biro_name"]
    ]
    .sort_values(["source_entity", "biro_code"])
    .drop_duplicates(
        subset=["source_entity", "biro_code"]
    )
    .reset_index(drop=True)
)

biro["entity_id"] = (
    biro["source_entity"].map(entity_lookup)
)

biro = create_ids(biro, "biro_id")

biro_lookup = {
    (row.source_entity, row.biro_code): row.biro_id
    for row in biro.itertuples(index=False)
}

biro = biro[
    ["biro_id", "entity_id", "biro_code", "biro_name"]
]


# 4. FUNDS_CENTER
fc_codes = pd.concat([
    budget["funds_center_code"],
    fc_source["target_code"]
]).dropna().drop_duplicates().sort_values()

fc_names = (
    budget.dropna(subset=["funds_center_code"])
    .drop_duplicates("funds_center_code")
    .set_index("funds_center_code")["funds_center_name"]
)

funds_center = pd.DataFrame({
    "funds_center_code": fc_codes.tolist()
})

funds_center["funds_center_name"] = (
    funds_center["funds_center_code"].map(fc_names)
)

funds_center = create_ids(
    funds_center, "funds_center_id"
)

fc_lookup = dict(zip(
    funds_center["funds_center_code"],
    funds_center["funds_center_id"]
))


# 5. COMMITMENT_ITEM
ci_codes = pd.concat([
    budget["commitment_item_code"],
    ci_source["target_code"]
]).dropna().drop_duplicates().sort_values()

ci_names = (
    budget.drop_duplicates("commitment_item_code")
    .set_index("commitment_item_code")["commitment_item_name"]
)

commitment_item = pd.DataFrame({
    "commitment_item_code": ci_codes.tolist()
})

commitment_item["commitment_item_name"] = (
    commitment_item["commitment_item_code"].map(ci_names)
)

commitment_item = create_ids(
    commitment_item, "commitment_item_id"
)

ci_lookup = dict(zip(
    commitment_item["commitment_item_code"],
    commitment_item["commitment_item_id"]
))


# 6. FUND
fund = pd.DataFrame({
    "fund_code": sorted(
        fund_source["target_code"].dropna().unique()
    )
})

fund = create_ids(fund, "fund_id")

fund_lookup = dict(zip(
    fund["fund_code"],
    fund["fund_id"]
))


# 7. MASTER_FC
master_fc = pd.DataFrame({
    "from_cost_center": fc_source["from_code"],
    "to_cost_center": fc_source["to_code"],
    "target_funds_center_id": (
        fc_source["target_code"].map(fc_lookup)
    )
})

master_fc = create_ids(master_fc, "master_fc_id")


# 8. MASTER_CI
master_ci = pd.DataFrame({
    "from_gl_account": ci_source["from_code"],
    "to_gl_account": ci_source["to_code"],
    "target_commitment_item_id": (
        ci_source["target_code"].map(ci_lookup)
    )
})

master_ci = create_ids(master_ci, "master_ci_id")


# 9. MASTER_FUND
master_fund = pd.DataFrame({
    "from_profit_center": fund_source["from_code"],
    "to_profit_center": fund_source["to_code"],
    "target_fund_id": (
        fund_source["target_code"].map(fund_lookup)
    )
})

master_fund = create_ids(master_fund, "master_fund_id")


# 10. BUDGET_MASTER
budget_master = pd.DataFrame({
    "entity_id": budget["source_entity"].map(entity_lookup),

    "biro_id": [
        biro_lookup.get((entity, code))
        for entity, code in zip(
            budget["source_entity"],
            budget["biro_code"]
        )
    ],

    "funds_center_id": (
        budget["funds_center_code"].map(fc_lookup)
    ),

    "commitment_item_id": (
        budget["commitment_item_code"].map(ci_lookup)
    ),

    "budget_scope": budget["budget_scope"],

    "current_budget": budget["current_budget"],
    "consumable_budget": budget["consumable_budget"],
    "consumed_budget": budget["consumed_budget"],
    "available_amount": budget["available_amount"]
})

budget_master = create_ids(
    budget_master, "budget_id"
)


# 11. USERS - TEMPLATE
users = pd.DataFrame(columns=[
    "user_id",
    "username",
    "password",
    "role",
    "entity_id",
    "funds_center_id"
])


# 12. BUDGET_REQUEST - TEMPLATE
budget_request = pd.DataFrame(columns=[
    "request_id",
    "user_id",
    "entity_id",
    "request_date",
    "pic",
    "input_profit_center",
    "input_cost_center",
    "input_gl_account",
    "request_amount",
    "request_description",
    "mapped_fund_id",
    "mapped_funds_center_id",
    "mapped_commitment_item_id",
    "budget_id",
    "budget_scope",
    "available_amount_snapshot",
    "shortage_amount",
    "approval_hold_amount_snapshot",
    "mapping_status",
    "budget_status",
    "request_status"
])


# 13. APPROVAL_DECISION - TEMPLATE
approval_decision = pd.DataFrame(columns=[
    "approval_id",
    "request_id",
    "approval_user_id",
    "decision",
    "instruction",
    "approved_amount",
    "decision_date"
])


# 14. BUDGET_ALLOCATION - TEMPLATE
budget_allocation = pd.DataFrame(columns=[
    "allocation_id",
    "request_id",
    "source_budget_id",
    "target_budget_id",
    "allocation_type",
    "amount",
    "instruction",
    "approved_by"
])


# 15. BUDGET_HISTORY - TEMPLATE
budget_history = pd.DataFrame(columns=[
    "history_id",
    "budget_id",
    "request_id",
    "action",
    "field_name",
    "previous_value",
    "updated_value",
    "amount",
    "created_by",
    "transaction_date"
])


# 16. FOREIGN KEY VALIDATION
def validate_fk(df, fk_col, ref_df, pk_col, nullable=False):

    if not nullable:
        assert df[fk_col].notna().all(), (
            f"NULL pada FK: {fk_col}"
        )

    actual = set(df[fk_col].dropna())
    valid = set(ref_df[pk_col].dropna())

    assert actual.issubset(valid), (
        f"Invalid FK: {fk_col}"
    )


validate_fk(
    biro, "entity_id",
    unit_entity, "entity_id"
)

validate_fk(
    master_fc, "target_funds_center_id",
    funds_center, "funds_center_id"
)

validate_fk(
    master_ci, "target_commitment_item_id",
    commitment_item, "commitment_item_id"
)

validate_fk(
    master_fund, "target_fund_id",
    fund, "fund_id"
)

validate_fk(
    budget_master, "entity_id",
    unit_entity, "entity_id"
)

validate_fk(
    budget_master, "biro_id",
    biro, "biro_id"
)

validate_fk(
    budget_master, "funds_center_id",
    funds_center, "funds_center_id",
    nullable=True
)

validate_fk(
    budget_master, "commitment_item_id",
    commitment_item, "commitment_item_id"
)

assert budget_master.loc[
    budget_master["budget_scope"] == "FC_CI",
    "funds_center_id"
].notna().all(), "FC_CI wajib mempunyai Funds Center"

# Entity pada Biro harus sesuai Entity Budget
biro_entity_lookup = dict(zip(
    biro["biro_id"], biro["entity_id"]
))

assert all(
    biro_entity_lookup[bid] == eid
    for bid, eid in zip(
        budget_master["biro_id"],
        budget_master["entity_id"]
    )
), "Entity Biro tidak sesuai dengan Entity Budget"

assert len(budget_master) == 5809

print("PK/FK VALIDATION PASSED")


# 17. EXPORT CSV
tables = {
    "unit_entity": unit_entity,
    "funds_center": funds_center,
    "commitment_item": commitment_item,
    "fund": fund,
    "biro": biro,
    "master_fc": master_fc,
    "master_ci": master_ci,
    "master_fund": master_fund,
    "budget_master": budget_master,
    "users": users,
    "budget_request": budget_request,
    "approval_decision": approval_decision,
    "budget_allocation": budget_allocation,
    "budget_history": budget_history
}

print("\n" + "=" * 65)
print("SUPABASE CSV EXPORT SUMMARY")
print("=" * 65)

for name, df in tables.items():

    path = OUTPUT_DIR / f"{name}.csv"

    df.to_csv(
        path,
        index=False,
        encoding="utf-8"
    )

    print(f"{name:<25} {len(df):>6,} rows")

print("\n" + "=" * 65)
print("TOTAL TABLES :", len(tables))
print("TOTAL BUDGET :", len(budget_master))
print("OUTPUT FOLDER:", OUTPUT_DIR)
print("=" * 65)

print("\nALL CSV FILES GENERATED SUCCESSFULLY")

PK/FK VALIDATION PASSED

SUPABASE CSV EXPORT SUMMARY
unit_entity                    7 rows
funds_center               1,652 rows
commitment_item            1,449 rows
fund                         112 rows
biro                          52 rows
master_fc                  1,776 rows
master_ci                  1,499 rows
master_fund                  334 rows
budget_master              5,809 rows
users                          0 rows
budget_request                 0 rows
approval_decision              0 rows
budget_allocation              0 rows
budget_history                 0 rows

TOTAL TABLES : 14
TOTAL BUDGET : 5809
OUTPUT FOLDER: data/supabase_import

ALL CSV FILES GENERATED SUCCESSFULLY


## STEP 17 — Buat User Dari Data Biro

**Tujuan:** Buat satu akun pengusul per kombinasi Entity dan Biro. Username pada langkah ini masih format awal.

In [17]:
# STEP 17 - GENERATE USERS BY BIRO NAME
import re
import unicodedata

# 1. Ambil data biro
df_users = biro[
    ["biro_id", "entity_id", "biro_code", "biro_name"]
].copy()

# 2. Fungsi membuat username
def make_username(text):
    text = str(text).lower().strip()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        c for c in text if not unicodedata.combining(c)
    )
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")

# 3. Username menggunakan nama biro
df_users["username"] = (
    df_users["biro_name"].apply(make_username)
)

# 4. Jika username sama, tambahkan Entity ID
duplicate_names = df_users["username"].duplicated(
    keep=False
)

df_users.loc[duplicate_names, "username"] = (
    df_users.loc[duplicate_names, "username"]
    + "_e"
    + df_users.loc[duplicate_names, "entity_id"].astype(str)
)

# 5. Jika masih ada username sama, tambahkan Biro ID
duplicate_names = df_users["username"].duplicated(
    keep=False
)

df_users.loc[duplicate_names, "username"] = (
    df_users.loc[duplicate_names, "username"]
    + "_b"
    + df_users.loc[duplicate_names, "biro_id"].astype(str)
)

# 6. Buat User ID dan role
df_users.insert(
    0, "user_id",
    range(1, len(df_users) + 1)
)

df_users["role"] = "Pengusul"
df_users["password"] = pd.NA
df_users["funds_center_id"] = pd.NA

# 7. Susun kolom sesuai ERD yang direvisi
df_users = df_users[
    [
        "user_id",
        "username",
        "password",
        "role",
        "entity_id",
        "biro_id",
        "funds_center_id"
    ]
]

# 8. Validasi
assert df_users["username"].is_unique
assert df_users["biro_id"].is_unique

# 9. Export CSV
df_users.to_csv(
    OUTPUT_DIR / "users.csv",
    index=False,
    encoding="utf-8"
)

print("TOTAL USERS:", len(df_users))
print("UNIQUE USERNAMES:", df_users["username"].nunique())

display(
    df_users.merge(
        biro[["biro_id", "biro_name"]],
        on="biro_id",
        how="left"
    )[
        ["username", "biro_name", "entity_id", "biro_id"]
    ].head(15)
)

TOTAL USERS: 52
UNIQUE USERNAMES: 52


,username,biro_name,entity_id,biro_id
0,biro_manajemen_rantai_pasok_e1,BIRO MANAJEMEN RANTAI PASOK,1,1
1,staf_analisis_evalua_e1,STAF ANALISIS EVALUA,1,2
2,biro_akuntansi_e1,BIRO AKUNTANSI,1,3
3,silog_log_ikn_e2,SILOG LOG IKN,2,4
4,general,GENERAL,3,5
5,biro_internal_audit,BIRO INTERNAL AUDIT,3,6
6,biro_pengadaan_barang_jasa_e3,BIRO PENGADAAN BARANG & JASA,3,7
7,seksi_pengadaan_logistik_fabrikasi,SEKSI PENGADAAN LOGISTIK & FABRIKASI,3,8
8,biro_hukum_manajemen_risiko,BIRO HUKUM & MANAJEMEN RISIKO,3,9
9,biro_pengembangan_perusahaan,BIRO PENGEMBANGAN PERUSAHAAN,3,10


## STEP 18 — Betulkan Format ID Di CSV Budget

**Tujuan:** Ubah ID yang sempat tertulis seperti `83.0` menjadi `83` agar cocok dengan kolom BIGINT di Supabase.

In [18]:
# step 18 - perbaiki csv budget master
# memperbaiki format kolom id pada budget_master
import pandas as pd
from pathlib import Path

file_path = Path("data/supabase_import/budget_master.csv")

# 1. Baca CSV
df_fix = pd.read_csv(file_path, dtype="string")

# 2. Kolom ID bertipe BIGINT pada Supabase
id_columns = [
    "budget_id",
    "entity_id",
    "biro_id",
    "funds_center_id",
    "commitment_item_id"
]

# 3. Konversi menjadi integer nullable
for col in id_columns:

    numeric_values = pd.to_numeric(
        df_fix[col],
        errors="raise"
    )

    # Pastikan tidak ada ID pecahan
    if ((numeric_values.dropna() % 1) != 0).any():
        raise ValueError(
            f"Kolom {col} memiliki ID pecahan!"
        )

    df_fix[col] = numeric_values.astype("Int64")

# 4. Simpan ulang CSV
df_fix.to_csv(
    file_path,
    index=False,
    encoding="utf-8",
    na_rep=""
)

# 5. Validasi hasil
print("=" * 55)
print("STEP 18 - FIX INTEGER FORMAT")
print("=" * 55)

print(f"Total baris: {len(df_fix):,}")

print("\nTipe data ID:")
print(df_fix[id_columns].dtypes)

print("\nContoh hasil:")
display(df_fix[id_columns].head(10))

print("\nCSV BERHASIL DIPERBAIKI")
print("Lokasi:", file_path)

STEP 18 - FIX INTEGER FORMAT
Total baris: 5,809

Tipe data ID:
budget_id             Int64
entity_id             Int64
biro_id               Int64
funds_center_id       Int64
commitment_item_id    Int64
dtype: object

Contoh hasil:


,budget_id,entity_id,biro_id,funds_center_id,commitment_item_id
0,1,3,5,<NA>,1163
1,2,3,5,<NA>,1177
2,3,3,5,<NA>,1246
3,4,3,5,<NA>,1251
4,5,3,5,<NA>,1252
5,6,3,5,<NA>,1257
6,7,3,5,<NA>,1260
7,8,3,5,<NA>,1262
8,9,3,5,<NA>,1263
9,10,3,5,<NA>,1264



CSV BERHASIL DIPERBAIKI
Lokasi: data/supabase_import/budget_master.csv


## STEP 19 — Rapikan Nama User

**Tujuan:** Ambil nama biro asli, rapikan penulisan, lalu tambahkan Entity kalau namanya sama.

In [19]:
# STEP 19 - CLEAN USERNAME FORMAT
import pandas as pd
import re
from pathlib import Path

OUTPUT_DIR = Path("data/supabase_import")

df_users = pd.read_csv(
    OUTPUT_DIR / "users.csv",
    dtype="string"
)

df_biro = pd.read_csv(
    OUTPUT_DIR / "biro.csv",
    dtype="string"
)

df_entity = pd.read_csv(
    OUTPUT_DIR / "unit_entity.csv",
    dtype="string"
)

# 1. Ambil nama biro dan entity asli
df = df_users.drop(
    columns=["biro_name", "unit_entity"],
    errors="ignore"
)

df = df.merge(
    df_biro[["biro_id", "biro_name"]],
    on="biro_id",
    how="left",
    validate="many_to_one"
)

df = df.merge(
    df_entity[["entity_id", "unit_entity"]],
    on="entity_id",
    how="left",
    validate="many_to_one"
)

assert df["biro_name"].notna().all()
assert df["unit_entity"].notna().all()

# 2. Singkatan yang harus tetap kapital
ACRONYMS = {
    "IKN", "SILOG", "SVP", "SPV",
    "SDM", "K3", "IT", "HR", "SHE",
    "VUBA", "VUDS", "VULS", "BUBI"
}

# Awalan unit kerja yang tidak perlu ditambah "Biro"
PREFIXES = (
    "biro", "staf", "seksi", "bagian",
    "department", "departemen",
    "divisi", "unit", "bidang"
)

# 3. Format nama unit kerja
def format_unit_name(name):

    name = re.sub(r"\s+", " ", str(name)).strip()

    words = []

    for word in name.split():
        if word.upper() in ACRONYMS:
            words.append(word.upper())
        else:
            words.append(word.capitalize())

    result = " ".join(words)

    # Pertahankan nama unit kerja yang sudah memiliki awalan
    if result.lower().startswith(PREFIXES):
        return result

    # Jangan tambah Biro untuk nama jabatan atau organisasi khusus
    special_words = [
        "SILOG", "SVP", "SPV", "Pallet"
    ]

    if any(
        result.lower().startswith(w.lower())
        for w in special_words
    ):
        return result

    return "Biro " + result


# 4. Generate username
df["username"] = df["biro_name"].apply(format_unit_name)

# 5. Tambahkan Entity hanya jika username duplikat
duplicates = df["username"].str.casefold().duplicated(
    keep=False
)

df.loc[duplicates, "username"] = (
    df.loc[duplicates, "username"]
    + " - "
    + df.loc[duplicates, "unit_entity"].str.upper()
)

# 6. Jika masih ada duplikasi, tambahkan kode biro
duplicates = df["username"].str.casefold().duplicated(
    keep=False
)

df.loc[duplicates, "username"] = (
    df.loc[duplicates, "username"]
    + " ("
    + df.loc[duplicates, "biro_code"].fillna(
        df.loc[duplicates, "biro_id"]
    )
    + ")"
) if "biro_code" in df.columns else df["username"]

# 7. Validasi
assert df["username"].str.casefold().is_unique
assert len(df) == len(df_users)

# 8. Simpan kolom sesuai Supabase
df_users_final = df[
    [
        "user_id",
        "username",
        "password",
        "role",
        "entity_id",
        "biro_id",
        "funds_center_id"
    ]
].copy()

df_users_final.to_csv(
    OUTPUT_DIR / "users.csv",
    index=False,
    encoding="utf-8"
)

# 9. Preview semua username
print("TOTAL USERS:", len(df_users_final))

display(
    df[
        [
            "username",
            "biro_name",
            "unit_entity"
        ]
    ]
)

print("\nUsername berhasil dirapikan!")

TOTAL USERS: 52


,username,biro_name,unit_entity
0,Biro Manajemen Rantai Pasok - BUBI,BIRO MANAJEMEN RANTAI PASOK,BUBI
1,Staf Analisis Evalua - BUBI,STAF ANALISIS EVALUA,BUBI
2,Biro Akuntansi - BUBI,BIRO AKUNTANSI,BUBI
3,SILOG Log IKN - IKN,SILOG LOG IKN,IKN
4,Biro General,GENERAL,KANTOR PUSAT SILOG
5,Biro Internal Audit,BIRO INTERNAL AUDIT,KANTOR PUSAT SILOG
6,Biro Pengadaan Barang & Jasa - KANTOR PUSAT SILOG,BIRO PENGADAAN BARANG & JASA,KANTOR PUSAT SILOG
7,Seksi Pengadaan Logistik & Fabrikasi,SEKSI PENGADAAN LOGISTIK & FABRIKASI,KANTOR PUSAT SILOG
8,Biro Hukum & Manajemen Risiko,BIRO HUKUM & MANAJEMEN RISIKO,KANTOR PUSAT SILOG
9,Biro Pengembangan Perusahaan,BIRO PENGEMBANGAN PERUSAHAAN,KANTOR PUSAT SILOG



Username berhasil dirapikan!


## STEP 20 — Sesuaikan CSV Users Terakhir

**Tujuan:** Hapus kolom `funds_center_id` dari CSV dan isi password contoh yang sebelumnya dipakai untuk uji lokal. **Jangan gunakan password bersama ini untuk login Supabase yang bisa diakses publik.**

In [20]:
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("data/supabase_import")

# Baca users.csv
df_users = pd.read_csv(
    OUTPUT_DIR / "users.csv",
    dtype="string"
)

# Hapus funds_center_id
df_users = df_users.drop(
    columns=["funds_center_id"],
    errors="ignore"
)

# Password testing
df_users["password"] = "pass123"

# Susun sesuai tabel users
df_users = df_users[
    [
        "user_id",
        "username",
        "password",
        "role",
        "entity_id",
        "biro_id"
    ]
]

# Simpan ulang
df_users.to_csv(
    OUTPUT_DIR / "users.csv",
    index=False,
    encoding="utf-8"
)

print("Total users:", len(df_users))
print("Password testing: pass123")
print("users.csv berhasil diperbarui!")

Total users: 52
Password testing: pass123
users.csv berhasil diperbarui!
